# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant JSON-LD schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install --quiet mlcroissant matplotlib seaborn

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset
dataset = mlc.Dataset(croissant_url)
# Access metadata object (not as dict)
metadata = dataset.metadata

# Print basic dataset information
print("=== Dataset Information ===")
print("Title:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Published:", metadata.datePublished)
print("Identifier:", metadata.identifier)
print("Cite As:", metadata.citeAs)
print("License:", metadata.license)


## 2. Data Overview
Review available record sets and fields, referencing all entities by their `@id`.

In [ ]:
# Get all available record sets from metadata
print("=== Available record sets ===")
record_sets = getattr(metadata, 'recordSet', [])
if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet @id: {rs['@id']}")
        if 'name' in rs:
            print(f"  Name: {rs['name']}")
        if 'description' in rs:
            print(f"  Description: {rs['description']}")
        # List fields for this record set
        fields = rs.get('field', [])
        if fields:
            print(f"  Fields:")
            for fld in fields:
                fid = fld.get('@id', None)
                fdesc = fld.get('description', None)
                print(f"    - {fid} | Description: {fdesc}")
        else:
            print("  No fields listed.")

# Default to print all records from the first record set (if present) and show the @id
print("\n=== Preview records from the first record set ===")
if record_sets:
    record_set_id = record_sets[0]['@id']
    for idx, x in enumerate(dataset.records(record_set=record_set_id)):
        print(f"Record {idx+1} (from {record_set_id}):")
        # Print only the first 3 for brevity
        print(x)
        if idx >= 2:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Reference each by its `@id`.

> Note: All fields, columns, and record sets are referenced by their `@id`.

In [ ]:
# Extract all record set IDs
record_set_ids = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]
dataframes = {}

for rs_id in record_set_ids:
    print(f"Extracting records from RecordSet: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    print(f"Columns for {rs_id}: {df.columns.tolist()}")
    print(f"First 3 rows for {rs_id}:")
    print(df.head(3))
    dataframes[rs_id] = df

# For EDA, select the largest record set (if present)
if dataframes:
    # Choose the record set with the most records
    main_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    print(f"\nWill use record set {main_record_set_id} for EDA.")
else:
    main_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply common processing steps: filtering by a numeric field, normalizing, grouping. Use fields exclusively by their `@id`.


In [ ]:
import numpy as np

# For demonstration, select a numeric field by @id. 
# Let's assume a field named 'age' is present with @id = 'age' (update if actual @id is different).

# Show all columns with their @id
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print("Columns (field @id) in DataFrame:")
    print(df.columns.tolist())

    # Guessing a numeric field (common in medical datasets)
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in [np.int64, np.float64]]
    if not possible_numeric_fields:
        possible_numeric_fields = [col for col in df.columns if pd.to_numeric(df[col], errors='coerce').notna().sum()/len(df[col]) > 0.5]

    # Pick first numeric field as demonstration
    if possible_numeric_fields:
        numeric_field_id = possible_numeric_fields[0]
        print(f"Using numeric field @id: {numeric_field_id}")

        threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
        filtered_df = df[pd.to_numeric(df[numeric_field_id], errors='coerce') > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        numeric_values = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df[numeric_field_id + '_normalized'] = (numeric_values - numeric_values.mean()) / numeric_values.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, numeric_field_id + '_normalized']].head())

        # Try grouping by a categorical field (e.g., anatomical location or sex)
        possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field_id]
        group_field_id = possible_group_fields[0] if possible_group_fields else None
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable grouping field found.")
    else:
        print("No numeric field detected for EDA.")
else:
    print("No record sets available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields.

We'll use `matplotlib` and `seaborn` for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and possible_numeric_fields:
    df_plot = dataframes[main_record_set_id]
    numeric_field_id = possible_numeric_fields[0]

    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(df_plot[numeric_field_id], errors='coerce').dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show boxplot
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df_plot)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Not enough data for visualization.")

## 6. Conclusion
This notebook demonstrated loading and exploration of the FAIR^2 colorectal cancer dataset using `mlcroissant`, referencing all entities by their `@id`. You can extend this workflow for deeper analysis, ensuring reproducibility and proper provenance tracking as enabled by the Croissant schema.